## Aggregate Table Creation
Builds summary/aggregate gold tables from gold.sentiment_scored

In [0]:
USE CATALOG ai_tool_sentiment;

-- 1. Daily sentiment by tool
-- The core trend table: how does sentiment for each tool move day to day.
CREATE OR REPLACE TABLE gold.sentiment_by_tool_daily AS
SELECT
    tool,
    DATE(created_at) AS post_date,
    COUNT(*) AS post_count,
    ROUND(AVG(sentiment_score), 3) AS avg_sentiment,
    ROUND(SUM(CASE WHEN sentiment_score > 0.2 THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS positive_pct,
    ROUND(SUM(CASE WHEN sentiment_score < -0.2 THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS negative_pct,
    ROUND(SUM(CASE WHEN sentiment_score BETWEEN -0.2 AND 0.2 THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS neutral_pct
FROM gold.sentiment_scored
WHERE sentiment_score IS NOT NULL
GROUP BY tool, DATE(created_at)
ORDER BY post_date DESC, tool;

-- 2. Sentiment by tool + source
-- Compares how a tool is perceived differently across Reddit vs Hacker News.
CREATE OR REPLACE TABLE gold.sentiment_by_tool_source AS
SELECT
    tool,
    source,
    COUNT(*) AS post_count,
    ROUND(AVG(sentiment_score), 3) AS avg_sentiment,
    ROUND(MIN(sentiment_score), 3) AS min_sentiment,
    ROUND(MAX(sentiment_score), 3) AS max_sentiment
FROM gold.sentiment_scored
WHERE sentiment_score IS NOT NULL
GROUP BY tool, source
ORDER BY tool, source;

-- 3. Overall tool leaderboard
-- Single ranked table: best-to-worst perceived tool overall, with volume
-- context (a tool with 5 mentions and 0.9 avg sentiment is less reliable
-- than one with 500 mentions and 0.4 avg sentiment).
CREATE OR REPLACE TABLE gold.tool_leaderboard AS
SELECT
    tool,
    COUNT(*) AS total_mentions,
    ROUND(AVG(sentiment_score), 3) AS avg_sentiment,
    ROUND(SUM(CASE WHEN sentiment_score > 0.2 THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS positive_pct,
    RANK() OVER (ORDER BY AVG(sentiment_score) DESC) AS sentiment_rank
FROM gold.sentiment_scored
WHERE sentiment_score IS NOT NULL
GROUP BY tool
ORDER BY avg_sentiment DESC;

-- 4. Weekly trend with week-over-week change
-- Shows momentum — is a tool's sentiment improving or declining recently.
CREATE OR REPLACE TABLE gold.sentiment_trend_weekly AS
WITH weekly AS (
    SELECT
        tool,
        DATE_TRUNC('WEEK', created_at) AS week_start,
        COUNT(*) AS post_count,
        ROUND(AVG(sentiment_score), 3) AS avg_sentiment
    FROM gold.sentiment_scored
    WHERE sentiment_score IS NOT NULL
    GROUP BY tool, DATE_TRUNC('WEEK', created_at)
)
SELECT
    tool,
    week_start,
    post_count,
    avg_sentiment,
    avg_sentiment - LAG(avg_sentiment) OVER (PARTITION BY tool ORDER BY week_start) AS sentiment_change_from_prior_week
FROM weekly
ORDER BY tool, week_start DESC;

-- 5. Top / bottom posts per tool (useful for dashboard drill-down or
-- "most positive/negative mention" callouts)
CREATE OR REPLACE TABLE gold.top_posts_by_tool AS
WITH ranked AS (
    SELECT
        tool,
        source,
        title,
        text,
        url,
        sentiment_score,
        created_at,
        ROW_NUMBER() OVER (PARTITION BY tool ORDER BY sentiment_score DESC) AS best_rank,
        ROW_NUMBER() OVER (PARTITION BY tool ORDER BY sentiment_score ASC) AS worst_rank
    FROM gold.sentiment_scored
    WHERE sentiment_score IS NOT NULL
)
SELECT tool, source, title, url, sentiment_score, created_at, 'most_positive' AS category
FROM ranked WHERE best_rank <= 5
UNION ALL
SELECT tool, source, title, url, sentiment_score, created_at, 'most_negative' AS category
FROM ranked WHERE worst_rank <= 5
ORDER BY tool, category, sentiment_score DESC;

-- Quick sanity check on all tables built
SELECT 'sentiment_by_tool_daily' AS table_name, COUNT(*) AS row_count FROM gold.sentiment_by_tool_daily
UNION ALL
SELECT 'sentiment_by_tool_source', COUNT(*) FROM gold.sentiment_by_tool_source
UNION ALL
SELECT 'tool_leaderboard', COUNT(*) FROM gold.tool_leaderboard
UNION ALL
SELECT 'sentiment_trend_weekly', COUNT(*) FROM gold.sentiment_trend_weekly
UNION ALL
SELECT 'top_posts_by_tool', COUNT(*) FROM gold.top_posts_by_tool;